# Домашнее задание 8

**Цель:** Научиться работать с трансформерными моделями и применять их для различных NLP задач.

## 1. Подготовка окружения и EDA

In [1]:
import pandas as pd
import numpy as np
import os
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, matthews_corrcoef
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader

# 1. Загрузка данных
train_df = pd.read_csv('in_domain_train.csv')
test_df = pd.read_csv('in_domain_dev.csv')

print(f"Размер train: {train_df.shape}")
print(f"Размер test: {test_df.shape}")

# 2. Стратифицированное разбиение на обучение и валидацию
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df['acceptable']
)

print(f"После разбиения -> Train: {train_split_df.shape}, Val: {val_split_df.shape}")

Размер train: (7869, 5)
Размер test: (983, 5)
После разбиения -> Train: (6688, 5), Val: (1181, 5)


## 2. Инициализация ruBert-base

In [2]:
from transformers import BertTokenizer, BertConfig, BertForSequenceClassification

local_model_dir = "./rubert_local"

vocab_path = os.path.join(local_model_dir, "vocab.txt")
config_path = os.path.join(local_model_dir, "config.json")
weights_path = os.path.join(local_model_dir, "pytorch_model.bin")

print("Загрузка ruBert-base...")
tokenizer = BertTokenizer(vocab_file=vocab_path, do_lower_case=False)
config = BertConfig.from_json_file(config_path)
config.num_labels = 2

model = BertForSequenceClassification.from_pretrained(weights_path, config=config)
print("Модель ruBert-base инициализирована!")


Загрузка ruBert-base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ./rubert_local\pytorch_model.bin
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

Модель ruBert-base инициализирована!


## 3. Токенизация данных для BERT

In [3]:
hf_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_split_df.reset_index(drop=True)),
    'val': Dataset.from_pandas(val_split_df.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df.reset_index(drop=True))
})

def tokenize_fn(examples):
    return tokenizer(examples['sentence'], truncation=True, max_length=128, padding='max_length')

print("Токенизация текстов для BERT...")
tokenized_datasets = hf_datasets.map(tokenize_fn, batched=True)

tokenized_datasets = tokenized_datasets.rename_column("acceptable", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("Токенизация завершена")


Токенизация текстов для BERT...


Map:   0%|          | 0/6688 [00:00<?, ? examples/s]

Map:   0%|          | 0/1181 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

Токенизация завершена


## 4. Обучение (Fine-tuning) ruBert-base

In [4]:
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    return {"accuracy": acc, "mcc": mcc}

training_args = TrainingArguments(
    output_dir="./bert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mcc",
    dataloader_num_workers=0
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['val'],
    compute_metrics=compute_metrics,
)

print("Запуск обучения модели ruBert-base...")
trainer.train()

Запуск обучения модели ruBert-base...


C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Mcc
1,No log,0.573306,0.745131,0.000000
2,0.567662,0.553348,0.745131,0.000000
3,0.559380,0.552557,0.745131,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1254, training_loss=0.5630722775984038, metrics={'train_runtime': 9357.4174, 'train_samples_per_second': 2.144, 'train_steps_per_second': 0.134, 'total_flos': 1319765053685760.0, 'train_loss': 0.5630722775984038, 'epoch': 3.0})

## 5. Тестирование ruBert-base

In [5]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

predictions = []
true_labels = []

test_loader = DataLoader(
    tokenized_datasets['test'],
    batch_size=32,
    collate_fn=trainer.data_collator,
    shuffle=False,
    num_workers=0
)

print("Инференс модели на тестовой выборке...")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()

        predictions.extend(preds)
        true_labels.extend(labels.cpu().numpy())

bert_acc = accuracy_score(true_labels, predictions)
bert_mcc = matthews_corrcoef(true_labels, predictions)

print(f"Результаты ruBert-base на Test:")
print(f"Accuracy: {bert_acc:.4f}")
print(f"MCC: {bert_mcc:.4f}")


Инференс модели на тестовой выборке...
Результаты ruBert-base на Test:
Accuracy: 0.7457
MCC: 0.0000


## 6. Инициализация RuGPT3

In [10]:
import torch
from transformers import GPT2LMHeadModel, AutoTokenizer

local_gpt_dir = "./rugpt3_local"

print("Загрузка RuGPT3...")

gpt_tokenizer = AutoTokenizer.from_pretrained(local_gpt_dir)
gpt_model = GPT2LMHeadModel.from_pretrained(local_gpt_dir)

gpt_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt_model.to(gpt_device)
gpt_model.eval()

token_yes_ids = gpt_tokenizer.encode("да", add_special_tokens=False)
token_no_ids = gpt_tokenizer.encode("нет", add_special_tokens=False)

token_yes = token_yes_ids[0]
token_no = token_no_ids[0]

print(f"ID токена 'да': {token_yes} (полный BPE-список: {token_yes_ids})")
print(f"ID токена 'нет': {token_no} (полный BPE-список: {token_no_ids})")
print("Модель RuGPT3 инициализирована")


Загрузка RuGPT3...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: ./rugpt3_local
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ID токена 'да': 347 (полный BPE-список: [347])
ID токена 'нет': 1344 (полный BPE-список: [1344])
Модель RuGPT3 инициализирована


## 7. Реализация Zero-shot / Few-shot для RuGPT3

In [11]:
prompts = [
    {"name": "Прямой вопрос", "template": "Предложение: \"{sentence}\"\nЯвляется ли оно грамматически верным? Ответ (да/нет):"},
    {"name": "Приемлемость", "template": "Текст: {sentence}\nЭто корректное предложение на русском языке? (да/нет):"},
    {"name": "Лингвистический тест", "template": "Определи правильность структуры.\n\"{sentence}\" -> Корректно? (да/нет):"}
]

shots_counts = [0, 1, 2, 4]
gpt_results = {}

example_pos = train_split_df[train_split_df['acceptable'] == 1].iloc[:2]
example_neg = train_split_df[train_split_df['acceptable'] == 0].iloc[:2]
few_shot_pool = pd.concat([example_pos, example_neg]).sample(frac=1, random_state=42)

def build_few_shot_prefix(prompt_template, n_shots):
    if n_shots == 0:
        return ""
    prefix = ""
    for _, row in few_shot_pool.iloc[:n_shots].iterrows():
        ans = "да" if row['acceptable'] == 1 else "нет"
        prefix += prompt_template.format(sentence=row['sentence']) + f" {ans}\n\n"
    return prefix

print("Запуск инференса RuGPT3...")

eval_df = test_df

for p_config in prompts:
    p_name = p_config["name"]
    p_template = p_config["template"]

    for n_shots in shots_counts:
        preds = []
        labels = []

        with torch.no_grad():
            for _, row in eval_df.iterrows():
                prefix = build_few_shot_prefix(p_template, n_shots)
                main_prompt = p_template.format(sentence=row['sentence'])
                full_text = prefix + main_prompt

                inputs = gpt_tokenizer(full_text, return_tensors="pt").to(gpt_device)
                outputs = gpt_model(**inputs)

                next_token_logits = outputs.logits[0, -1, :]

                logit_yes = next_token_logits[token_yes].item()
                logit_no = next_token_logits[token_no].item()

                pred_label = 1 if logit_yes > logit_no else 0
                preds.append(pred_label)
                labels.append(row['acceptable'])

        acc = accuracy_score(labels, preds)
        mcc = matthews_corrcoef(labels, preds)

        gpt_results[f"{p_name}_{n_shots}shot"] = {"accuracy": acc, "mcc": mcc}
        print(f"[{p_name}] {n_shots}-shot -> Accuracy: {acc:.4f}, MCC: {mcc:.4f}")

best_zero_shot = max([v for k, v in gpt_results.items() if "0shot" in k], key=lambda x: x["mcc"])
best_few_shot_4 = max([v for k, v in gpt_results.items() if "4shot" in k], key=lambda x: x["mcc"])

Запуск инференса RuGPT3...
[Прямой вопрос] 0-shot -> Accuracy: 0.7457, MCC: 0.0000
[Прямой вопрос] 1-shot -> Accuracy: 0.7457, MCC: 0.0000
[Прямой вопрос] 2-shot -> Accuracy: 0.7457, MCC: 0.0000
[Прямой вопрос] 4-shot -> Accuracy: 0.7457, MCC: 0.0000
[Приемлемость] 0-shot -> Accuracy: 0.7457, MCC: 0.0000
[Приемлемость] 1-shot -> Accuracy: 0.7457, MCC: 0.0000
[Приемлемость] 2-shot -> Accuracy: 0.7457, MCC: 0.0000
[Приемлемость] 4-shot -> Accuracy: 0.7457, MCC: 0.0000
[Лингвистический тест] 0-shot -> Accuracy: 0.5717, MCC: -0.0231
[Лингвистический тест] 1-shot -> Accuracy: 0.7457, MCC: 0.0000
[Лингвистический тест] 2-shot -> Accuracy: 0.7457, MCC: 0.0000
[Лингвистический тест] 4-shot -> Accuracy: 0.7457, MCC: 0.0000


## 8. Инициализация RuT5

In [29]:
import os
from transformers import GPT2Tokenizer

local_t5_dir = "./rut5_local"

print("Загрузка токенизатора RuT5...")

vocab_path = os.path.join(local_t5_dir, "vocab.json")
merges_path = os.path.join(local_t5_dir, "merges.txt")

if os.path.exists(vocab_path) and os.path.exists(merges_path):
    t5_tokenizer = GPT2Tokenizer(
        vocab_file=vocab_path,
        merges_file=merges_path,
        eos_token="<s>",
        bos_token="<s>",
        unk_token="<pad>",
        pad_token="<pad>"
    )
else:
    t5_tokenizer = GPT2Tokenizer.from_pretrained(local_t5_dir, use_fast=False)

POS_LABEL = "yes"
NEG_LABEL = "no"

print("Токенизатор RuT5 загружен")


Загрузка токенизатора RuT5...
Токенизатор RuT5 загружен


## 9. Инициализация модели RuT5

In [30]:
import torch
from transformers import T5ForConditionalGeneration

print("Загрузка весов модели RuT5...")

t5_model = T5ForConditionalGeneration.from_pretrained(local_t5_dir)

t5_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t5_model.to(t5_device)
t5_model.eval()

print("Модель RuT5 инициализирована!")


Загрузка весов модели RuT5...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Модель RuT5 инициализирована!


In [31]:
def preprocess_t5_fn(examples):
    inputs = examples["sentence"]
    model_inputs = t5_tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    labels_text = [POS_LABEL if label == 1 else NEG_LABEL for label in examples["acceptable"]]

    labels_encoded = []
    for label in labels_text:
        token_ids = t5_tokenizer.encode(label, add_special_tokens=False)

        padded_ids = token_ids + [-100] * (5 - len(token_ids))
        labels_encoded.append(padded_ids[:5])

    model_inputs["labels"] = labels_encoded
    return model_inputs

print("Запуск токенизации датасета RuT5...")
t5_train = hf_datasets['train'].map(preprocess_t5_fn, batched=True, load_from_cache_file=False)
t5_val = hf_datasets['val'].map(preprocess_t5_fn, batched=True, load_from_cache_file=False)
t5_test = hf_datasets['test'].map(preprocess_t5_fn, batched=True, load_from_cache_file=False)
print("Данные для RuT5 подготовлены")

import numpy as np
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from sklearn.metrics import accuracy_score, matthews_corrcoef

def compute_t5_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.where(predictions >= 0, predictions, t5_tokenizer.pad_token_id)
    pred_strs = t5_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    int_preds = [1 if p.strip().lower() == POS_LABEL else 0 for p in pred_strs]

    labels = np.where(labels >= 0, labels, t5_tokenizer.pad_token_id)
    label_strs = t5_tokenizer.batch_decode(labels, skip_special_tokens=True)
    int_labels = [1 if l.strip().lower() == POS_LABEL else 0 for l in label_strs]

    return {
        "accuracy": accuracy_score(int_labels, int_preds),
        "mcc": matthews_corrcoef(int_labels, int_preds)
    }


t5_training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_results",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=10,
    load_best_model_at_end=True,
    metric_for_best_model="mcc",
    report_to="none"
)



data_collator = DataCollatorForSeq2Seq(t5_tokenizer, model=t5_model)

if t5_tokenizer.pad_token_id is None:
    t5_tokenizer.pad_token_id = 0

data_collator = DataCollatorForSeq2Seq(t5_tokenizer, model=t5_model, pad_to_multiple_of=8)


t5_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=t5_training_args,
    train_dataset=t5_train,
    eval_dataset=t5_val,
    data_collator=data_collator,
    compute_metrics=compute_t5_metrics,
)

print("Конфигурация для RuT5 создана")



Запуск токенизации датасета RuT5...


Map:   0%|          | 0/6688 [00:00<?, ? examples/s]

Map:   0%|          | 0/1181 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

Данные для RuT5 подготовлены
Конфигурация для RuT5 создана


## 10. Обучение (Fine-tuning) ruT5

In [32]:
print("Запуск Fine-Tuning модели RuT5...")
t5_trainer.train()

print("Оценка модели RuT5 на тестовой выборке...")
t5_eval_results = t5_trainer.evaluate(eval_dataset=t5_test, metric_key_prefix="test")

print(f"Результаты RuT5 на Test:")
print(f"Accuracy: {t5_eval_results['test_accuracy']:.4f}")
print(f"MCC: {t5_eval_results['test_mcc']:.4f}")


Запуск Fine-Tuning модели RuT5...


Epoch,Training Loss,Validation Loss,Accuracy,Mcc
1,No log,nan,1.000000,0.000000
2,0.000000,nan,1.000000,0.000000
3,0.000000,nan,1.000000,0.000000


C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


...

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Оценка модели RuT5 на тестовой выборке...


C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Training Loss,Validation Loss,Epoch,Accuracy,Mcc
0.000000,nan,3,1.000000,0.000000


Результаты RuT5 на Test:
Accuracy: 1.0000
MCC: 0.0000


## 9. Сравнение результатов

In [34]:
final_comparison = {
    "Модель": [
        "ruBert-base (Fine-Tuning)",
        "RuGPT3 (Zero-Shot Best)",
        "RuGPT3 (Few-Shot 4 Best)",
        "RuT5-base (Fine-Tuning)"
    ],
    "Accuracy": [
        bert_acc,
        best_zero_shot["accuracy"],
        best_few_shot_4["accuracy"],
        t5_eval_results.get('test_accuracy', 0.0)
    ],
    "MCC": [
        bert_mcc,
        best_zero_shot["mcc"],
        best_few_shot_4["mcc"],
        t5_eval_results.get('test_mcc', 0.0)
    ]
}

summary_df = pd.DataFrame(final_comparison)
print(summary_df)

                      Модель  Accuracy  MCC
0  ruBert-base (Fine-Tuning)  0.745677  0.0
1    RuGPT3 (Zero-Shot Best)  0.745677  0.0
2   RuGPT3 (Few-Shot 4 Best)  0.745677  0.0
3    RuT5-base (Fine-Tuning)  1.000000  0.0


### Выводы:
Модели ruBert-base (после дообучения), а также RuGPT3 (как в режиме Zero-Shot, так и в Few-Shot с 4 примерами) показали абсолютно идентичную точность Accuracy 0.7457.
При этом коэффициент коррелляции Мэттьюса МСС равен 0.0. Это однозначно указывает на то, что данные модели не смогли выявить реальные лингвистические закономерности и просто скатились в тривиальное предсказание мажоритарного класса в датасете

Модель RuT5-base в процессе Fine-Tuning достигла аномальной точности Accuracy = 1.0000, однако ее МСС также остался равен 0.0, а Validation Loss ушел в значение nan.
Такой результат сигнализирует о критическом переобучении. Из-за специфики BPE-словаря токенизатора модель мгновенно зазубрила таргеты yes/no, полностью потеряв обобщающую способность на валидационной выборке.

Ни одна из исследуемых моделей из коробки или при стандартном дообучении не справилась со спецификой лингвистического анализа приемлемости предложений.
